In [1]:
import numpy as np
import pandas as pd
import yaml

# ---------------------------------
# Display settings (for debugging)
# ---------------------------------
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# ---------------------------------
# Load settings (paths, years, etc.)
# ---------------------------------
with open('../../Settings.yaml', 'r') as file:
    Setting = yaml.safe_load(file)

# A container to store yearly energy sheets before concatenation
Data = []

# ============================================================
# 1) Build Water Usage panel (1383–1400) from multiple files
#    Each raw file is in wide format (years as columns) -> melt to long format
# ============================================================

# -----------------------------
# 1.1) Water usage: 1383–1393 (sheet 8)
# -----------------------------
file_path_1 = f"{Setting['Raw_Path']}/Y1383_Y1393_10AndMore_Water.xlsx"
T8 = pd.read_excel(file_path_1, sheet_name=8, skiprows=3)

# Years available in this file (wide columns)
years_1 = [str(y) for y in range(1383, 1394)]  # 1383..1393

# Assign consistent column names
T8.columns = ["Province_Code", "Province"] + years_1

# Convert from wide (years as columns) to long (Year, Water)
T8 = T8.melt(
    id_vars=["Province_Code", "Province"],
    value_vars=years_1,
    var_name="Year",
    value_name="Water (M3)"
)

# -----------------------------
# 1.2) Water usage: 1394–1399 (sheet 8)
# -----------------------------
file_path_2 = f"{Setting['Raw_Path']}/Y1394_Y1399_10AndMore_Water.xlsx"
T8_1 = pd.read_excel(file_path_2, sheet_name=8, skiprows=3)

years_2 = [str(y) for y in range(1394, 1400)]  # 1394..1399
T8_1.columns = ["Province_Code", "Province"] + years_2

T8_1 = T8_1.melt(
    id_vars=["Province_Code", "Province"],
    value_vars=years_2,
    var_name="Year",
    value_name="Water (M3)"
)

# -----------------------------
# 1.3) Water usage: 1400 (sheet 9)
# -----------------------------
file_path_3 = f"{Setting['Raw_Path']}/Y1400_10AndMore_Water.xlsx"
T9 = pd.read_excel(file_path_3, sheet_name=9, skiprows=3)

years_3 = ["1400"]
T9.columns = ["Province_Code", "Province"] + years_3

T9 = T9.melt(
    id_vars=["Province_Code", "Province"],
    value_vars=years_3,
    var_name="Year",
    value_name="Water (M3)"
)

# -----------------------------
# 1.4) Stack all water frames together
# -----------------------------
Water_Usage = pd.concat([T8, T8_1, T9], axis=0, ignore_index=True)

# Ensure correct dtypes and panel-friendly ordering
Water_Usage["Year"] = Water_Usage["Year"].astype(int)
Water_Usage["Province_Code"] = pd.to_numeric(Water_Usage["Province_Code"], errors="coerce")
Water_Usage = Water_Usage.sort_values(["Province_Code", "Year"]).reset_index(drop=True)

# Remove national total row
Water_Usage = Water_Usage[Water_Usage["Province"] != "کل کشور"]


# ============================================================
# 2) Read yearly Energy Excel files and build the base Energy dataset
#    Note: years 1381, 1382, 1401 are not available in raw files
# ============================================================

for year in range(Setting["Start_Year"] + 2, Setting["End_Year"]):
    file_name = f"Y{year}_10AndMore_Energy.xlsx"
    file_path = f"{Setting['Raw_Path']}/{file_name}"

    # Read energy table (sheet 12) and skip header artifacts
    T12 = pd.read_excel(file_path, sheet_name=12, skiprows=1)

    # Standardize column names to a consistent schema
    cols = [
        "Province",
        "Natural_Gas (M3)", "Fueloil (Liter)", "Gasoil (Liter)", "Gasoline (Liter)",
        "Kerosene (Liter)", "LNG (KG)", "Charcoal and Coal (KG)", "Electricity (KWH)"
    ]
    T12.columns = cols

    # Drop the first data row (often contains totals or non-data content)
    T12 = T12.iloc[1:].reset_index(drop=True)

    # Add the year identifier
    T12["Year"] = year

    # Store yearly data for later concatenation
    Data.append(T12)

# Concatenate all years into one long dataset
Energy_Usage = pd.concat(Data, ignore_index=True)


# ============================================================
# 3) Merge Energy and Water datasets (left join keeps all energy rows)
# ============================================================

Dataset = pd.merge(
    Energy_Usage,
    Water_Usage,
    how="left",
    on=["Year", "Province"]
)

# Remove national total rows
Dataset = Dataset[Dataset["Province"] != "کل کشور"]

# Normalize Arabic characters to Persian (to avoid merge/key mismatches)
Dataset["Province"] = (
    Dataset["Province"].astype(str)
    .str.replace("ي", "ی", regex=False)
    .str.replace("ك", "ک", regex=False)
    .str.replace("چهارمحال وبختیاری", "چهارمحال و بختیاری", regex=False)
)

# ============================================================
# 4) Add Province codes and create missing years (1381, 1382, 1401)
#    Then impute missing-year values using CAGR by province and variable
# ============================================================

# Columns to be imputed for missing years
energy_cols = [
    "Natural_Gas (M3)", "Fueloil (Liter)", "Gasoil (Liter)", "Gasoline (Liter)",
    "Kerosene (Liter)", "LNG (KG)", "Charcoal and Coal (KG)",
    "Electricity (KWH)", "Water (M3)"
]

# Years missing from raw energy files (to be generated)
missing_years = [1381, 1382, 1401]

# Load province codes/names mapping
input_file = f"{Setting['Province_Codes']}"
provinces = pd.read_excel(input_file, sheet_name="Province_Codes")

# Ensure province code has consistent type
provinces["Province_Code"] = provinces["Province_Code"].astype("float64")

# Ensure Year is numeric
Dataset["Year"] = pd.to_numeric(Dataset["Year"], errors="coerce")

# Attach province code to the Dataset (right join keeps Dataset rows)
Dataset = pd.merge(provinces, Dataset, how="right", on="Province")

# Cleanup duplicated province code column names after merge
Dataset.drop(columns={"Province_Code_y"}, inplace=True)
Dataset.rename(columns={"Province_Code_x": "Province_Code"}, inplace=True)

# Reorder columns to place Year first
Dataset = Dataset[["Year"] + [c for c in Dataset.columns if c != "Year"]]

# Create a full Province x MissingYears grid to append later
add_rows = provinces.merge(pd.DataFrame({"Year": missing_years}), how="cross")

# Initialize the imputed columns as NaN (will be filled by CAGR extrapolation)
for c in energy_cols:
    add_rows[c] = np.nan

# Convert energy columns to numeric to ensure clean calculations
for c in energy_cols:
    Dataset[c] = pd.to_numeric(Dataset[c], errors="coerce")

# ------------------------------
# CAGR-based extrapolation logic (per province, per energy variable):
# 1) Find first and last positive observations
# 2) Compute CAGR
# 3) Backcast for 1381, 1382 and forecast for 1401
# ------------------------------
for pro_code in add_rows["Province_Code"].unique():
    # Extract historical data for this province
    base = Dataset[Dataset["Province_Code"] == pro_code].copy()
    base = base.sort_values("Year")

    # Rows to fill in add_rows for this province
    add_mask = (add_rows["Province_Code"] == pro_code)
    years_to_add = add_rows.loc[add_mask, "Year"].tolist()

    for col in energy_cols:
        # Keep only valid (non-missing) and strictly positive values
        s = base[["Year", col]].dropna(subset=["Year", col]).sort_values("Year")
        s = s[s[col] > 0]

        # Need at least two points to estimate growth rate
        if len(s) < 2:
            continue

        # First and last observed year/value
        y0 = int(s.iloc[0]["Year"])
        v0 = float(s.iloc[0][col])
        y1 = int(s.iloc[-1]["Year"])
        v1 = float(s.iloc[-1][col])

        # Guard against invalid cases
        if y1 == y0 or v0 <= 0 or v1 <= 0:
            continue

        # CAGR computation
        cagr = (v1 / v0) ** (1 / (y1 - y0)) - 1

        # Extrapolate missing years
        vals = []
        for y in years_to_add:
            y = int(y)
            if y < y0:
                # Backcast from first observation
                vals.append(v0 / ((1 + cagr) ** (y0 - y)))
            elif y > y1:
                # Forecast from last observation
                vals.append(v1 * ((1 + cagr) ** (y - y1)))
            else:
                # (Not expected here) interpolate inside observed range
                vals.append(v0 * ((1 + cagr) ** (y - y0)))

        add_rows.loc[add_mask, col] = vals

# Append imputed rows to the Dataset
Dataset = pd.concat([Dataset, add_rows], ignore_index=True)

# Enforce unique Province-Year and sort for panel structure
Dataset = (
    Dataset
    .drop_duplicates(subset=["Province_Code", "Year"], keep="first")
    .sort_values(["Year", "Province_Code"])
    .reset_index(drop=True)
)

# Fill any remaining NaNs with 0 (NOTE: this may be a strong assumption for econometrics)
Dataset = Dataset.fillna(0)

# Drop any "total" province codes (if present in codes file)
Dataset = Dataset[Dataset["Province_Code"] != 0]


# ============================================================
# 5) Convert fuels to BOE and compute total fuel consumption (BOE)
# ============================================================

# Fuel columns to be converted to BOE
fuel_cols = [
    "Natural_Gas (M3)", "Fueloil (Liter)", "Gasoil (Liter)", "Gasoline (Liter)",
    "Kerosene (Liter)", "LNG (KG)", "Charcoal and Coal (KG)","Electricity (KWH)"
]

# BOE conversion factors (boe per unit)
boe_factor = {
    "Natural_Gas (M3)": 0.00609,             # boe per m3
    "Fueloil (Liter)": 0.00000662,           # boe per liter
    "Gasoil (Liter)": 0.00000585,            # boe per liter
    "Gasoline (Liter)": 0.00000540,          # boe per liter
    "Kerosene (Liter)": 0.00000600,          # boe per liter
    "LNG (KG)": 0.00817,                     # boe per kg
    "Charcoal and Coal (KG)": 0.00408,       # boe per kg
    "Electricity (KWH)": 0.000588            # boe per kwh
}

# Compute BOE per fuel and then sum across fuels
for c in fuel_cols:
    Dataset[f"{c} (boe)"] = Dataset[c] * boe_factor[c]

Dataset["Energy (boe)"] = Dataset[[f"{c} (boe)" for c in fuel_cols]].sum(axis=1)

# Drop intermediate BOE columns (keep only Energy and Electricity (boe))
Dataset.drop(
    columns=[f"{c} (boe)" for c in fuel_cols if c != "Electricity (KWH)"],
    inplace=True,
    errors="ignore"
)
Dataset.rename(columns={'Electricity (KWH) (boe)':'Electricity (boe)'},inplace=True)

Dataset['Elec_boe_intensity'] = Dataset['Electricity (boe)'] / Dataset['Energy (boe)']

# ============================================================
# 6) Export to Excel
# ============================================================

output_file_name = "Unadjusted.xlsx"
sheet_name = "Energy_By_Province"
path = f"{Setting['Output_Path_Unajusted']}/{output_file_name}"

# Write/replace the sheet in an existing workbook (mode="a")
with pd.ExcelWriter(path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    Dataset.to_excel(writer, sheet_name=sheet_name, index=False)

c:\Users\hwi\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
c:\Users\hwi\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
c:\Users\hwi\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
c:\Users\hwi\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
c:\Users\hwi\AppData\Local\Progr